# Hurst Exponent Estimation via Randomized Kolmogorov–Smirnov

This notebook estimates the Hurst exponent of log-realized volatility using a Randomized Kolmogorov–Smirnov approach.

The objective is to assess whether volatility exhibits rough behaviour, corresponding to estimated Hurst exponents below 0.5.

The analysis uses the cleaned per-index datasets produced by `00_data_preparation.ipynb`.

In [1]:
# ============================================================
# HURST EXPONENT ESTIMATION — RANDOMIZED KS
# Master's Thesis: From Fractional to Rough Volatility
# Author: Elisa Battista
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from itertools import combinations
from scipy.stats import ks_2samp
from google.colab import drive

# ------------------------------------------------------------
# Google Drive Setup
# ------------------------------------------------------------

MOUNTPOINT = "/content/drive"

if not os.path.isdir(f"{MOUNTPOINT}/MyDrive"):
    drive.mount(MOUNTPOINT)

BASE = f"{MOUNTPOINT}/MyDrive/thesis"
CLEAN_DIR = f"{BASE}/data/clean"
RESULTS_DIR = f"{BASE}/results/metrics"
FIG_DIR = f"{BASE}/results/figures"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

SUMMARY_CSV = f"{RESULTS_DIR}/hurst_randomizedKS_summary.csv"
LATEX_TEX = f"{RESULTS_DIR}/hurst_randomizedKS_table.tex"

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TAUS = [1, 5, 22]
H_GRID = np.round(np.arange(0.05, 0.40 + 1e-9, 0.01), 3)

N_RANDOMIZATIONS = 60
RANDOM_STATE = 42

RESET_OUTPUT_FILES = True

# ------------------------------------------------------------
# Helper Functions
# ------------------------------------------------------------

def log(message):
    print(f"[RKS-Hurst] {message}")


def discover_clean_files(clean_dir):
    """
    Discover per-index clean CSV files produced by data preparation.
    """
    paths = sorted(glob.glob(os.path.join(clean_dir, "*.csv")))
    files = {}

    for path in paths:
        try:
            head = pd.read_csv(path, nrows=10)

            if "Symbol" in head.columns and "Date" in head.columns:
                symbol = str(head["Symbol"].dropna().iloc[0])
                files[symbol] = path

        except Exception:
            pass

    return files


def read_log_volatility(path):
    """
    Read log-volatility from a clean per-index CSV.
    Prefer the stored log_vol column; otherwise compute it from rv5_ss.
    """
    df = pd.read_csv(path)

    if "Date" not in df.columns:
        raise ValueError("Missing required column: Date")

    df = df.sort_values("Date").reset_index(drop=True)

    if "log_vol" in df.columns:
        series = df["log_vol"].astype(float).to_numpy()

    elif "rv5_ss" in df.columns:
        series = 0.5 * np.log(df["rv5_ss"].astype(float).to_numpy())

    else:
        raise ValueError("Input CSV must contain either 'log_vol' or 'rv5_ss'.")

    series = series[np.isfinite(series)]

    return series


def make_increments(series, tau):
    """
    Compute increments of log-volatility at lag tau.
    """
    return series[tau:] - series[:-tau]


def normalize_increments(increments, tau, H):
    """
    Normalize increments according to the scaling tau^H.
    """
    return increments / (tau ** H)


def ks_objective(series, taus, H, rng):
    """
    Compute the maximum pairwise Kolmogorov-Smirnov distance
    between normalized increment distributions across time scales.
    """
    normalized_sets = []

    for tau in taus:
        increments = make_increments(series, tau)

        if increments.size == 0:
            return np.inf

        z = normalize_increments(increments, tau, H)

        # Random permutation reduces the effect of serial ordering
        z = z[rng.permutation(z.size)]

        normalized_sets.append(z)

    common_length = min(map(len, normalized_sets))

    if common_length < 50:
        return np.inf

    normalized_sets = [z[:common_length] for z in normalized_sets]

    max_distance = 0.0

    for i, j in combinations(range(len(normalized_sets)), 2):
        statistic, _ = ks_2samp(
            normalized_sets[i],
            normalized_sets[j],
            alternative="two-sided",
            mode="auto"
        )

        max_distance = max(max_distance, float(statistic))

    return max_distance


def estimate_hurst_randomized_ks(series):
    """
    Estimate H by minimizing the KS objective over a grid.
    The procedure is repeated over several randomizations and
    the final estimate is the median of the minimizers.
    """
    rng_master = np.random.default_rng(RANDOM_STATE)

    h_samples = []
    ks_samples = []

    for _ in range(N_RANDOMIZATIONS):
        rng = np.random.default_rng(
            rng_master.integers(0, 2**31 - 1)
        )

        objective_values = np.array([
            ks_objective(series, TAUS, H, rng)
            for H in H_GRID
        ])

        if not np.isfinite(objective_values).any():
            continue

        best_index = int(np.nanargmin(objective_values))

        h_samples.append(float(H_GRID[best_index]))
        ks_samples.append(float(objective_values[best_index]))

    if not h_samples:
        raise RuntimeError("KS objective is not finite anywhere on the grid.")

    h_samples = np.array(h_samples)
    ks_samples = np.array(ks_samples)

    ci_low, ci_high = np.percentile(h_samples, [2.5, 97.5])

    return {
        "H_hat": float(np.median(h_samples)),
        "KS_min": float(np.median(ks_samples)),
        "CI_low": float(ci_low),
        "CI_high": float(ci_high),
        "N_randomizations": int(len(h_samples)),
    }


def write_latex_table(results_df, output_path):
    """
    Write a compact LaTeX table with Hurst estimates.
    """
    with open(output_path, "w") as file:
        file.write("\\begin{table}[H]\n")
        file.write("\\centering\n")
        file.write("\\caption{Hurst exponent estimates via Randomized Kolmogorov--Smirnov.}\n")
        file.write("\\label{tab:hurst_rks}\n")
        file.write("\\begin{tabular}{lcccc}\n")
        file.write("\\toprule\n")
        file.write("Symbol & $\\widehat H$ & KS$_{\\min}$ & 95\\% CI & $n$ \\\\\n")
        file.write("\\midrule\n")

        for _, row in results_df.iterrows():
            ci = f"[{row['CI_95_low']:.3f}, {row['CI_95_high']:.3f}]"
            file.write(
                f"{row['Symbol']} & "
                f"{row['H_hat_KS']:.3f} & "
                f"{row['KS_min']:.4f} & "
                f"{ci} & "
                f"{int(row['N_obs'])} \\\\\n"
            )

        file.write("\\bottomrule\n")
        file.write("\\end{tabular}\n")
        file.write("\\end{table}\n")


# ------------------------------------------------------------
# Run Estimation
# ------------------------------------------------------------

if RESET_OUTPUT_FILES:
    for path in [SUMMARY_CSV, LATEX_TEX]:
        if os.path.exists(path):
            os.remove(path)
            log(f"Removed existing file: {path}")

files = discover_clean_files(CLEAN_DIR)

if not files:
    raise FileNotFoundError(f"No clean CSV files found in: {CLEAN_DIR}")

rows = []

log("Starting Hurst exponent estimation.")

for symbol, path in sorted(files.items()):

    log(f"Processing {symbol}")

    series = read_log_volatility(path)

    if len(series) < max(TAUS) + 100:
        log(f"Skipping {symbol}: series too short.")
        continue

    result = estimate_hurst_randomized_ks(series)

    rows.append({
        "Symbol": symbol,
        "H_hat_KS": result["H_hat"],
        "KS_min": result["KS_min"],
        "CI_95_low": result["CI_low"],
        "CI_95_high": result["CI_high"],
        "N_randomizations": result["N_randomizations"],
        "N_obs": len(series),
    })

    log(
        f"{symbol}: "
        f"H={result['H_hat']:.3f}, "
        f"KS={result['KS_min']:.4f}, "
        f"CI=[{result['CI_low']:.3f}, {result['CI_high']:.3f}]"
    )

results_df = (
    pd.DataFrame(rows)
    .sort_values("Symbol")
    .reset_index(drop=True)
)

results_df.to_csv(SUMMARY_CSV, index=False)
write_latex_table(results_df, LATEX_TEX)

display(results_df)

log(f"Saved CSV: {SUMMARY_CSV}")
log(f"Saved LaTeX table: {LATEX_TEX}")
log("Hurst estimation completed successfully.")

Mounted at /content/drive
[RKS-Hurst] Removed existing file: /content/drive/MyDrive/thesis/results/metrics/hurst_randomizedKS_summary.csv
[RKS-Hurst] Removed existing file: /content/drive/MyDrive/thesis/results/metrics/hurst_randomizedKS_table.tex
[RKS-Hurst] Starting Hurst exponent estimation.
[RKS-Hurst] Processing .FTSE
[RKS-Hurst] .FTSE: H=0.120, KS=0.0251, CI=[0.110, 0.120]
[RKS-Hurst] Processing .HSI
[RKS-Hurst] .HSI: H=0.080, KS=0.0176, CI=[0.080, 0.080]
[RKS-Hurst] Processing .IXIC
[RKS-Hurst] .IXIC: H=0.150, KS=0.0187, CI=[0.140, 0.150]
[RKS-Hurst] Processing .N225
[RKS-Hurst] .N225: H=0.140, KS=0.0179, CI=[0.140, 0.140]
[RKS-Hurst] Processing .SPX
[RKS-Hurst] .SPX: H=0.140, KS=0.0256, CI=[0.140, 0.150]
[RKS-Hurst] Processing .STOXX50E
[RKS-Hurst] .STOXX50E: H=0.170, KS=0.0308, CI=[0.170, 0.180]


,Symbol,H_hat_KS,KS_min,CI_95_low,CI_95_high,N_randomizations,N_obs
0,.FTSE,0.12,0.025130,0.11,0.12,60,4638
1,.HSI,0.08,0.017602,0.08,0.08,60,4510
2,.IXIC,0.15,0.018724,0.14,0.15,60,4615
3,.N225,0.14,0.017949,0.14,0.14,60,4479
4,.SPX,0.14,0.025560,0.14,0.15,60,4619
5,.STOXX50E,0.17,0.030842,0.17,0.18,60,4691


[RKS-Hurst] Saved CSV: /content/drive/MyDrive/thesis/results/metrics/hurst_randomizedKS_summary.csv
[RKS-Hurst] Saved LaTeX table: /content/drive/MyDrive/thesis/results/metrics/hurst_randomizedKS_table.tex
[RKS-Hurst] Hurst estimation completed successfully.
